# DuckDB and Dask side by side, on a synthetic release-shaped dataset

Synthetic data only -- shaped like `packages/catalog`'s real `releases`
table (see `packages/catalog/src/networked_players_catalog/discogs/parquet.py`)
but generated in-notebook, not read from a real ingest. For a real dataset,
see `02-explore-real-discogs-catalog.ipynb`.


In [7]:
import random
from pathlib import Path

import pandas as pd

random.seed(0)
countries = ["US", "GB", "DE", "JP", "FR", None]
quality = ["Correct", "Needs Vote", "Complete and Correct", None]

rows = [
    {
        "release_id": i,
        "title": f"Synthetic Release {i}",
        "country": random.choice(countries),
        "released": f"{random.randint(1960, 2025)}",
        "data_quality": random.choice(quality),
    }
    for i in range(50_000)
]
pdf = pd.DataFrame(rows)

out_path = Path("/tmp/synthetic_releases.parquet")
pdf.to_parquet(out_path, index=False)
print(f"Wrote {len(pdf)} synthetic rows to {out_path}")

Wrote 50000 synthetic rows to /tmp/synthetic_releases.parquet


In [8]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE VIEW releases AS SELECT * FROM '{out_path}'")
con.sql("SELECT country, count(*) n FROM releases GROUP BY 1 ORDER BY 2 DESC").show()

┌─────────┬───────┐
│ country │   n   │
│ varchar │ int64 │
├─────────┼───────┤
│ DE      │  8471 │
│ FR      │  8400 │
│ NULL    │  8358 │
│ GB      │  8281 │
│ JP      │  8247 │
│ US      │  8243 │
└─────────┴───────┘



In [9]:
import dask.dataframe as dd
from dask.distributed import Client

client = Client("tcp://dask-scheduler:8786")

# from_pandas (not read_parquet on out_path) deliberately: out_path is a
# local file inside this Jupyter container only, not on a shared filesystem
# the dask-worker containers can see. from_pandas scatters the in-memory
# partitions to workers over the network instead, which is the right
# approach here -- see 02-explore-real-discogs-catalog.ipynb for the
# real-Parquet-on-a-shared-mount case.
ddf = dd.from_pandas(pdf, npartitions=8)
print("Same aggregate, computed distributed across the real worker(s):")
print(ddf.groupby("country").size().compute().sort_values(ascending=False))

Same aggregate, computed distributed across the real worker(s):
country
DE    8471
FR    8400
GB    8281
JP    8247
US    8243
dtype: int64
